# MODULE 43 — OBJECT SEGMENTATION

# 6.2 U-Net — Encoder–Decoder Architecture

**Must Know · Semantic Segmentation**

U-Net is one of the most important architectures in image segmentation, particularly in medical imaging.

The central idea behind U-Net is simple:

> The encoder learns what is present in an image, while the decoder learns where each object is located. Skip connections combine both types of information to produce accurate segmentation masks.

Let's understand how it works internally. 🚀

---

## 1. Why do we need U-Net?

Recall FCN.

FCN extracts features using convolutional layers, produces class-score maps and upsamples them to the original resolution.

However, downsampling loses spatial details.

Consider a medical image containing a small tumour.

![U-Net based glioma segmentation](https://www.mdpi.com/cancers/cancers-14-04457/article_deploy/html/images/cancers-14-04457-g001.png)

Source: [U-Net Based Segmentation and Characterization of Gliomas](https://www.mdpi.com/2072-6694/14/18/4457)

The model needs to identify:

- Where the tumour is located.
- Its exact boundaries.
- Its shape and size.
- Which pixels belong to the tumour.

If the model only uses deep, low-resolution features, recovering precise boundaries becomes difficult.

U-Net addresses this through skip connections between its encoder and decoder.

## 2. What is U-Net?

U-Net is a fully convolutional encoder–decoder architecture introduced for biomedical image segmentation.

It has three major components:

1. Contracting path (Encoder)
2. Bottleneck
3. Expanding path (Decoder)

![U-Net architecture](https://miro.medium.com/v2/1%2A0qJdnSxkerwZPCWVdLPpHw.png)

Source: [U-Net Architecture Explained](https://lukmanaj.medium.com/u-net-architecture-revolutionizing-computer-vision-through-innovative-image-segmentation-e1e2155c38b1)

Its structure resembles the letter U, which explains its name.

### Overall architecture

```text
                  INPUT IMAGE
                  256 × 256
                       |
           ┌───────────┴───────────┐
           |                       |
        ENCODER                SKIP FEATURES
           |                       |
       Downsampling                |
           |                       |
           ↓                       |
        ENCODER                    |
           |                       |
       Downsampling                |
           |                       |
           ↓                       |
        BOTTLENECK                 |
           |                       |
       Upsampling                  |
           |                       |
           ↓                       |
        DECODER  ←─────────────────┘
           |
       Upsampling
           |
           ↓
        DECODER
           |
       Upsampling
           |
           ↓
       OUTPUT MASK
```

The diagram is conceptual; actual U-Net connects corresponding encoder and decoder stages through concatenation.

## 3. Encoder — Contracting Path

**Feature Extraction**

The encoder extracts increasingly complex features from the input image.

It typically consists of:

- Two 3×3 convolution layers.
- ReLU activation after each convolution.
- 2×2 max pooling for downsampling.

![U-Net encoder and decoder structure](https://gyokhan.com/content/images/2025/11/data-src-image-30124467-6f4c-4845-988a-c7fe08ff9ae1.png)

Source: [U-Net Architecture — Complete Diagram](https://www.gyokhan.com/simple-u-net-explaination/)

### Step 1 — Convolution

Convolution extracts features from the input image.

Early layers learn:

- Edges
- Corners
- Textures

Deeper layers learn:

- Shapes
- Patterns
- Object-level information

### Step 2 — ReLU activation

ReLU introduces non-linearity.

$$
ReLU(x)=\max(0,x)
$$

### Step 3 — Max Pooling

U-Net commonly uses 2×2 max pooling with stride 2.

This reduces spatial dimensions by half.

For example:

```text
Input:
256 × 256 × 3

After Encoder Block 1:
256 × 256 × 64

After Max Pooling:
128 × 128 × 64
```

As the encoder progresses, spatial resolution decreases while feature channels increase.

### Typical encoder progression

For a common four-level U-Net:

| Stage | Feature map |
|---|---|
| Input | 256 × 256 × 3 |
| Encoder 1 | 256 × 256 × 64 |
| Encoder 2 | 128 × 128 × 128 |
| Encoder 3 | 64 × 64 × 256 |
| Encoder 4 | 32 × 32 × 512 |
| Bottleneck | 16 × 16 × 1024 |

These are illustrative dimensions for a common configuration, not mandatory dimensions for every U-Net implementation.

**Important observation:**

- Height and width decrease.
- Number of channels increases.
- The receptive field increases.

The encoder learns increasingly abstract representations of the image.

## 4. Bottleneck — Bridge Between Encoder and Decoder

The bottleneck is the deepest part of U-Net.

![U-Net bottleneck and architecture](https://miro.medium.com/v2/1%2A0qJdnSxkerwZPCWVdLPpHw.png)

Source: [U-Net Architecture](https://lukmanaj.medium.com/u-net-architecture-revolutionizing-computer-vision-through-innovative-image-segmentation-e1e2155c38b1)

For our example:

```text
Input to Bottleneck:
16 × 16 × 512

After Convolutions:
16 × 16 × 1024
```

It contains high-level contextual information extracted by the encoder.

Think of it this way:

- Encoder: Understands the image.
- Bottleneck: Holds deep contextual features.
- Decoder: Reconstructs spatial predictions.

The bottleneck passes its features to the decoder.

## 5. Decoder — Expanding Path

**Spatial Reconstruction**

The decoder gradually restores the spatial resolution of the feature maps.

It typically uses:

1. Upsampling or transposed convolution.
2. Skip connection concatenation.
3. Convolution layers and ReLU.

![U-Net decoder and skip connections](https://gyokhan.com/content/images/2025/11/data-src-image-30124467-6f4c-4845-988a-c7fe08ff9ae1.png)

Source: [U-Net Architecture — Complete Diagram](https://www.gyokhan.com/simple-u-net-explaination/)

### Step 1 — Upsampling

The decoder increases the spatial dimensions.

For example:

```text
16 × 16 × 1024
       ↓
  Upsampling
       ↓
32 × 32 × 512
```

U-Net implementations commonly use either transposed convolution or interpolation followed by convolution.

### Step 2 — Skip Connection

This is the most important part of U-Net.

The decoder receives feature maps from the corresponding encoder stage.

We concatenate these feature maps along the channel dimension.

We'll understand this in detail in the next section.

### Step 3 — Convolution

After concatenation, convolution layers process the combined features.

This helps the decoder reconstruct object boundaries and spatial details.

### Typical decoder progression

| Stage | Feature map after fusion and convolution |
|---|---|
| Decoder 1 | 32 × 32 × 512 |
| Decoder 2 | 64 × 64 × 256 |
| Decoder 3 | 128 × 128 × 128 |
| Decoder 4 | 256 × 256 × 64 |

## 6. Skip Connections — The Heart of U-Net

**Extremely Important**

Why do we need skip connections when we already have an encoder and decoder?

Let's understand the problem.

During encoding, pooling reduces spatial resolution.

This helps capture high-level contextual information but loses fine spatial details.

During decoding, upsampling restores spatial dimensions, but it cannot automatically recover all the information lost during downsampling.

Skip connections transfer high-resolution encoder features directly to the decoder.

![U-Net skip connection architecture](https://gyokhan.com/content/images/2025/11/data-src-image-30124467-6f4c-4845-988a-c7fe08ff9ae1.png)

Source: [U-Net Architecture](https://www.gyokhan.com/simple-u-net-explaination/)

### How does concatenation work?

Suppose we have:

**Encoder feature map:**

```python
[B, 256, 64, 64]
```

**Upsampled decoder feature map:**

```python
[B, 256, 64, 64]
```

Both have matching spatial dimensions.

We concatenate them along the channel dimension:

```python
import torch

encoder_features = torch.randn(1, 256, 64, 64)
decoder_features = torch.randn(1, 256, 64, 64)

combined = torch.cat(
    [encoder_features, decoder_features],
    dim=1
)

print(combined.shape)
```

**Output:**

```text
torch.Size([1, 512, 64, 64])
```

Why 512 channels?

$$
256+256=512
$$

The decoder now receives:

- Earlier encoder features containing spatial details.
- Decoder features containing deeper contextual information.

The subsequent convolution layers learn how to combine these features.

### Why is this useful?

Imagine segmenting a small tumour.

The encoder may identify the region as containing abnormal tissue, while earlier feature maps retain more precise boundary information.

The decoder combines both to produce a better mask.

**Remember:** U-Net uses concatenation-based skip connections in its original architecture, unlike architectures that commonly use element-wise addition.

## 7. Final Segmentation Output

After the final decoder stage, we obtain a high-resolution feature map.

A 1×1 convolution converts its channels into class-score channels.

![U-Net segmentation output example](https://swanlab-docs-1301372061.cos.ap-beijing.myqcloud.com/assets/zh/examples/unet-medical-segmentation/results2.png)

Source: [U-Net Medical Image Segmentation Example](https://docs.swanlab.cn/examples/unet-medical-segmentation.html)

Suppose we have three classes:

```text
0 → Background
1 → Healthy tissue
2 → Tumour
```

Our final output shape:

```text
[B, 3, 256, 256]
```

This means every pixel has three class scores.

We apply `argmax` over the class dimension to obtain the predicted segmentation mask:

```python
predicted_mask = logits.argmax(dim=1)

print(predicted_mask.shape)
```

**Output:**

```text
[B, 256, 256]
```

## 8. How U-Net Is Trained

U-Net requires image-mask pairs.

![Medical image and segmentation mask](https://www.mdpi.com/cancers/cancers-14-04457/article_deploy/html/images/cancers-14-04457-g001.png)

Source: [U-Net Based Segmentation and Characterization of Gliomas](https://www.mdpi.com/2072-6694/14/18/4457)

For example:

| Input | Ground truth |
|---|---|
| MRI image | Tumour mask |
| Road image | Road segmentation mask |
| Leaf image | Disease region mask |

The model predicts a mask, which is compared with the ground truth.

### Common loss functions

**1. Cross Entropy Loss**

Commonly used for multi-class segmentation.

$$
L_{CE}=-\sum_{c=1}^{C}y_c\log(p_c)
$$

**2. Dice Loss**

Useful when overlap quality matters, particularly with imbalanced foreground and background regions.

$$
L_{Dice}=1-\frac{2|P\cap G|+\epsilon}{|P|+|G|+\epsilon}
$$

Where:

- \(P\) = predicted mask.
- \(G\) = ground-truth mask.
- \(\epsilon\) = small constant for numerical stability.

A combined loss is also common:

$$
L = L_{CE}+L_{Dice}
$$

The appropriate loss depends on the task and mask representation.

## 9. U-Net Variants

You should recognize these names, although the original U-Net is the priority.

![U-Net architecture variants](https://miro.medium.com/v2/1%2A0qJdnSxkerwZPCWVdLPpHw.png)

**Original U-Net**

Encoder–decoder with concatenation-based skip connections. Introduced for biomedical image segmentation.

**3D U-Net**

Uses 3D convolutions to process volumetric data, such as MRI and CT scans.

**Attention U-Net**

Uses attention gates to emphasize relevant spatial features.

**U-Net++**

Uses nested and redesigned skip pathways to improve feature fusion.

## 10. U-Net vs FCN

This comparison is important.

| Feature | FCN | U-Net |
|---|---|---|
| Architecture | Fully convolutional | Encoder–decoder |
| Feature extraction | CNN backbone | Contracting path |
| Upsampling | Upsampling layers | Expanding path |
| Skip connections | FCN-16s and FCN-8s | Core architectural feature |
| Feature fusion | Score-map fusion | Feature-map concatenation |
| Common application | General semantic segmentation | Biomedical and detailed segmentation |

## 11. Industry Applications

![U-Net segmentation examples](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AFAT6nMejnyo2pZ3OB4lVEg.png)

Source: [Image Segmentation Using U-Net](https://medium.com/data-science/image-segmentation-unet-and-deep-supervision-loss-using-keras-model-f21a9856750a)

| Industry | Application |
|---|---|
| Healthcare | Tumour, organ and tissue segmentation |
| Automotive | Road and lane segmentation |
| Manufacturing | Surface defect segmentation |
| Agriculture | Crop and leaf disease segmentation |
| Satellite imaging | Building and land-cover segmentation |

---

## What you should remember 🧠

1. U-Net consists of an encoder, bottleneck and decoder.
2. Encoder extracts increasingly complex features through convolution and pooling.
3. Bottleneck captures deep contextual information.
4. Decoder restores spatial resolution through upsampling.
5. Skip connections transfer encoder features to corresponding decoder stages.
6. Concatenation combines spatial details and contextual features.
7. A final 1×1 convolution generates pixel-level class scores.

---